# FIFA Dataset EDA and Feature Validation

This notebook provides an interactive EDA and feature-engineering validation workflow for the prepared FIFA dataset.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'ML').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current notebook path.')

repo_root = find_repo_root(Path.cwd())
players_clean_path = repo_root / 'ML' / 'data' / 'processed' / 'players_clean.csv'
model_ready_path = repo_root / 'ML' / 'data' / 'features' / 'model_ready.csv'

players_clean = pd.read_csv(players_clean_path)
model_ready = pd.read_csv(model_ready_path)

print('players_clean path:', players_clean_path)
print('model_ready path:', model_ready_path)

players_clean path: /home/adham/Desktop/DEPI-ML-PROJECT/ML/data/processed/players_clean.csv
model_ready path: /home/adham/Desktop/DEPI-ML-PROJECT/ML/data/features/model_ready.csv


In [2]:
print('players_clean shape:', players_clean.shape)
print('model_ready shape:', model_ready.shape)

display(players_clean.head(3))
display(model_ready.head(3))

missing_overview = pd.DataFrame({
    'dataset': ['players_clean', 'model_ready'],
    'total_nulls': [int(players_clean.isna().sum().sum()), int(model_ready.isna().sum().sum())],
})
display(missing_overview)

players_clean shape: (20621, 15)
model_ready shape: (20621, 18)


,short_name,age,overall,potential,player_positions,position_group,market_value_eur,pace,shooting,passing,dribbling,defending,physic,age_potential_gap,value_per_rating
0,A. Abaz,20,52,65,GK,GK,170000.0,69.0,55.0,58.0,63.0,56.0,66.0,13,3207.547170
1,A. Abdallah,24,64,69,"LB, LWB, LM",DEF,825000.0,79.0,37.0,55.0,62.0,58.0,66.0,5,12692.307692
2,A. Abdennour,32,65,65,CB,DEF,475000.0,49.0,53.0,56.0,51.0,65.0,71.0,0,7196.969697


,age,overall,potential,age_potential_gap,pace,shooting,passing,dribbling,defending,physic,position_group_GK,position_group_DEF,position_group_MID,position_group_ATT,log_value,short_name,market_value_eur,value_per_rating
0,20,52,65,13,69.0,55.0,58.0,63.0,56.0,66.0,1,0,0,0,12.043560,A. Abaz,170000.0,3207.547170
1,24,64,69,5,79.0,37.0,55.0,62.0,58.0,66.0,0,1,0,0,13.623140,A. Abdallah,825000.0,12692.307692
2,32,65,65,0,49.0,53.0,56.0,51.0,65.0,71.0,0,1,0,0,13.071072,A. Abdennour,475000.0,7196.969697


,dataset,total_nulls
0,players_clean,0
1,model_ready,0


In [3]:
numeric_cols_clean = players_clean.select_dtypes(include=[np.number]).columns
numeric_summary_clean = players_clean[numeric_cols_clean].describe().T
numeric_summary_clean['skew'] = players_clean[numeric_cols_clean].skew()
display(numeric_summary_clean)

,count,mean,std,min,25%,50%,75%,max,skew
age,20621.0,2.507391e+01,4.823808e+00,16.000000,21.000,25.000000,2.900000e+01,4.400000e+01,0.439044
overall,20621.0,6.541487e+01,6.845382e+00,46.000000,61.000,65.000000,7.000000e+01,9.100000e+01,0.082533
potential,20621.0,7.056864e+01,6.173581e+00,48.000000,66.000,70.000000,7.500000e+01,9.500000e+01,0.177636
market_value_eur,20621.0,2.523742e+06,5.941294e+06,9000.000000,475000.000,925000.000000,1.900000e+06,5.050000e+07,5.395462
pace,20621.0,6.830183e+01,1.015952e+01,28.000000,63.000,69.000000,7.500000e+01,9.700000e+01,-0.633740
shooting,20621.0,5.275249e+01,1.297737e+01,16.000000,44.000,55.000000,6.200000e+01,9.100000e+01,-0.377175
passing,20621.0,5.728859e+01,9.323488e+00,25.000000,52.000,58.000000,6.300000e+01,9.300000e+01,-0.189901
dribbling,20621.0,6.247607e+01,8.926104e+00,28.000000,58.000,63.000000,6.800000e+01,9.400000e+01,-0.525659
defending,20621.0,5.206653e+01,1.520482e+01,15.000000,40.000,56.000000,6.300000e+01,9.000000e+01,-0.539682
physic,20621.0,6.469512e+01,9.203150e+00,30.000000,59.000,66.000000,7.100000e+01,9.000000e+01,-0.494944


In [4]:
position_counts = players_clean['position_group'].value_counts().sort_index()
position_distribution = (position_counts / position_counts.sum()).rename('share')
display(pd.concat([position_counts.rename('count'), position_distribution], axis=1))

position_summary = (
    players_clean.groupby('position_group', as_index=True)
    .agg(
        count=('short_name', 'count'),
        age_mean=('age', 'mean'),
        overall_mean=('overall', 'mean'),
        potential_mean=('potential', 'mean'),
        market_value_median=('market_value_eur', 'median'),
        market_value_mean=('market_value_eur', 'mean'),
    )
    .sort_index()
)
display(position_summary)

,count,share
position_group,,
ATT,4053,0.196547
DEF,6870,0.333156
GK,2311,0.112070
MID,7387,0.358227


,count,age_mean,overall_mean,potential_mean,market_value_median,market_value_mean
position_group,,,,,,
ATT,4053,24.792499,65.563040,70.924500,975000.0,2.993016e+06
DEF,6870,25.379039,65.641776,70.372052,900000.0,2.316485e+06
GK,2311,25.836002,63.968412,69.448723,500000.0,1.667033e+06
MID,7387,24.706105,65.575064,70.906593,1000000.0,2.727036e+06


In [5]:
position_dummy_cols = [
    'position_group_GK',
    'position_group_DEF',
    'position_group_MID',
    'position_group_ATT',
]

age_gap_expected = players_clean['potential'] - players_clean['overall']
age_gap_mismatch = int((players_clean['age_potential_gap'] != age_gap_expected).sum())

value_per_rating_expected = players_clean['market_value_eur'] / (players_clean['overall'] + 1.0)
value_per_rating_mismatch = int(((players_clean['value_per_rating'] - value_per_rating_expected).abs() > 1e-9).sum())
value_per_rating_nonfinite = int((~np.isfinite(players_clean['value_per_rating'])).sum())

missing_dummy_cols = [c for c in position_dummy_cols if c not in model_ready.columns]
invalid_one_hot_rows = None
if not missing_dummy_cols:
    invalid_one_hot_rows = int((model_ready[position_dummy_cols].sum(axis=1) != 1).sum())

checks = pd.DataFrame([
    {'check': 'age_potential_gap mismatch rows', 'value': age_gap_mismatch},
    {'check': 'value_per_rating mismatch rows', 'value': value_per_rating_mismatch},
    {'check': 'value_per_rating non-finite rows', 'value': value_per_rating_nonfinite},
    {'check': 'one-hot missing columns count', 'value': len(missing_dummy_cols)},
    {'check': 'one-hot invalid rows', 'value': invalid_one_hot_rows if invalid_one_hot_rows is not None else -1},
])

display(checks)
if missing_dummy_cols:
    print('Missing one-hot columns:', missing_dummy_cols)

,check,value
0,age_potential_gap mismatch rows,0
1,value_per_rating mismatch rows,0
2,value_per_rating non-finite rows,0
3,one-hot missing columns count,0
4,one-hot invalid rows,0


In [6]:
corr_source = model_ready.select_dtypes(include=[np.number])
corr_with_log_value = (
    corr_source.corr(numeric_only=True)['log_value']
    .drop(labels=['log_value'])
    .dropna()
    .sort_values(key=np.abs, ascending=False)
)
display(corr_with_log_value.to_frame('corr_with_log_value').head(12))

,corr_with_log_value
overall,0.882689
potential,0.839302
value_per_rating,0.723550
market_value_eur,0.702702
dribbling,0.602831
passing,0.588193
shooting,0.397034
physic,0.336088
pace,0.275528
defending,0.217803


In [8]:
import json

def iqr_outlier_stats(series: pd.Series) -> pd.Series:
    valid = series.dropna()
    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((valid < lower) | (valid > upper)).sum())
    return pd.Series({
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': outlier_count,
        'outlier_share': outlier_count / len(valid) if len(valid) else 0.0,
    })

def quantile_tail_stats(series: pd.Series, lower_q: float = 0.01, upper_q: float = 0.99) -> pd.Series:
    valid = series.dropna()
    lower = valid.quantile(lower_q)
    upper = valid.quantile(upper_q)
    below = int((valid < lower).sum())
    above = int((valid > upper).sum())
    return pd.Series({
        'lower_q': lower_q,
        'upper_q': upper_q,
        'lower_bound': lower,
        'upper_bound': upper,
        'below_count': below,
        'above_count': above,
        'tail_share': (below + above) / len(valid) if len(valid) else 0.0,
    })

preprocessing_report_path = repo_root / 'ML' / 'data' / 'processed' / 'preprocessing_report.json'
with preprocessing_report_path.open('r', encoding='utf-8') as f:
    preprocessing_report = json.load(f)

winsorization_table = pd.DataFrame.from_dict(
    preprocessing_report.get('winsorization', {}),
    orient='index',
)
display(winsorization_table)

outlier_table = pd.DataFrame({
    'market_value_eur_raw': iqr_outlier_stats(players_clean['market_value_eur']),
    'market_value_log1p': iqr_outlier_stats(np.log1p(players_clean['market_value_eur'])),
    'log_value': iqr_outlier_stats(model_ready['log_value']),
    'overall': iqr_outlier_stats(players_clean['overall']),
    'potential': iqr_outlier_stats(players_clean['potential']),
    'age': iqr_outlier_stats(players_clean['age']),
}).T
display(outlier_table)

tail_table = pd.DataFrame({
    'market_value_eur_raw': quantile_tail_stats(players_clean['market_value_eur']),
    'market_value_log1p': quantile_tail_stats(np.log1p(players_clean['market_value_eur'])),
    'log_value': quantile_tail_stats(model_ready['log_value']),
}).T
display(tail_table)

print('Interpretation: raw market value is naturally heavy-tailed, so IQR flags many valid elite players. Use log_value and the winsorization table for preprocessing decisions.')

,lower_quantile,upper_quantile,lower_bound,upper_bound,lower_clipped_rows,upper_clipped_rows,total_clipped_rows
market_value_eur,0.01,0.99,80000.0,35000000.0,163,205,368


,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_share
market_value_eur_raw,475000.000000,1.900000e+06,1.425000e+06,-1.662500e+06,4.037500e+06,2274.0,0.110276
market_value_log1p,13.071072,1.445736e+01,1.386293e+00,1.099163e+01,1.653680e+01,784.0,0.038019
log_value,13.071072,1.445736e+01,1.386293e+00,1.099163e+01,1.653680e+01,784.0,0.038019
overall,61.000000,7.000000e+01,9.000000e+00,4.750000e+01,8.350000e+01,184.0,0.008923
potential,66.000000,7.500000e+01,9.000000e+00,5.250000e+01,8.850000e+01,92.0,0.004461
age,21.000000,2.900000e+01,8.000000e+00,9.000000e+00,4.100000e+01,4.0,0.000194


,lower_q,upper_q,lower_bound,upper_bound,below_count,above_count,tail_share
market_value_eur_raw,0.01,0.99,80000.000000,3.500000e+07,163.0,205.0,0.017846
market_value_log1p,0.01,0.99,11.289794,1.737086e+01,163.0,205.0,0.017846
log_value,0.01,0.99,11.289794,1.737086e+01,163.0,205.0,0.017846


Interpretation: raw market value is naturally heavy-tailed, so IQR flags many valid elite players. Use log_value and the winsorization table for preprocessing decisions.


## Notes

- This notebook mirrors the automated script at ML/scripts/generate_eda_report.py.
- Use this notebook for interactive inspection, and the script for reproducible report generation.
- For outliers, rely on log_value diagnostics plus preprocessing_report.json winsorization metadata; raw market_value_eur IQR counts are expected to look high because of a valid heavy tail.